estructura csv:

periodico | titulo | texto | url

In [1]:
!pip install requests beautifulsoup4 pandas tqdm transformers sentencepiece -q

In [2]:
import requests
import pandas as pd
import time
import random
import json
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from tqdm import tqdm


START_URL = "https://www.tercerainformacion.es/opinion/"
OUTPUT_CSV = "data_tercerainformacion_opinion.csv"

MAX_ITEMS = 10
MAX_PAGES = 300
MIN_DELAY = 1.5
MAX_DELAY = 3.0
TIMEOUT = 20

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) "
        "Gecko/20100101 Firefox/150.0"
    ),
    "Accept-Language": "es-ES,es;q=0.9",
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,"
        "image/avif,image/webp,*/*;q=0.8"
    ),
    "Connection": "keep-alive",
}


def limpiar_texto(txt):
    if not txt:
        return ""
    return " ".join(txt.replace("\n", " ").replace("\t", " ").split())


def dormir():
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))


def pedir_sopa(url):
    r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)

    if r.status_code in [403, 429]:
        raise RuntimeError(
            f"El sitio devolvió {r.status_code}. "
            "Se detiene para evitar forzar el acceso."
        )

    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def es_url_tercera(url):
    try:
        netloc = urlparse(url).netloc.lower()
        return netloc.endswith("tercerainformacion.es")
    except Exception:
        return False


def parece_articulo_opinion(url):
    path = urlparse(url).path.lower()

    if not es_url_tercera(url):
        return False

    # Los artículos de opinión suelen ir dentro de /opinion/
    if not path.startswith("/opinion/"):
        return False

    # Evita páginas de paginación/listado
    if "/page/" in path:
        return False

    if any(x in path for x in [
        "/tag/",
        "/tags/",
        "/author/",
        "/autor/",
        "/autores/",
        "/category/",
        "/wp-content/",
        "/feed/",
        "/rss",
        "/xml",
        "/comentarios",
        "/contacto",
        "/colabora",
        "/anunciate",
        "/informacion-legal",
        "/politica-privacidad",
        "/cookies"
    ]):
        return False

    # Evita la portada /opinion/
    partes = [p for p in path.split("/") if p]
    if len(partes) < 2:
        return False

    return True


def es_basura(txt):
    txt_lower = txt.lower()

    basura = [
        "whatsapp",
        "facebook",
        "twitter",
        "x.com",
        "linkedin",
        "telegram",
        "newsletter",
        "suscríbete",
        "suscribete",
        "iniciar sesión",
        "inicia sesión",
        "regístrate",
        "mostrar comentarios",
        "lee también",
        "lo más leído",
        "última hora",
        "publicidad",
        "aviso legal",
        "política de privacidad",
        "política de cookies",
        "copyright",
        "recibe las noticias",
        "todas las noticias publicadas",
        "compartir",
        "comparte",
        "síguenos",
        "seguir leyendo",
        "te puede interesar",
        "otras noticias",
        "newsletter diaria",
        "boletín",
        "comentarios",
        "normas de participación",
        "personaliza tus cookies",
        "una publicación de",
        "queda prohibida toda reproducción",
        "tercera información",
        "margenblanco",
        "estudio nexos"
    ]

    return any(b in txt_lower for b in basura)


def extraer_urls_listado(soup, page_url):
    articulos = []
    vistos = set()

    for a in soup.find_all("a", href=True):
        url = urljoin(page_url, a["href"])
        titulo = limpiar_texto(a.get_text(" ", strip=True))

        if len(titulo) < 8:
            continue

        if not parece_articulo_opinion(url):
            continue

        if url in vistos:
            continue

        vistos.add(url)

        articulos.append({
            "titulo_listado": titulo,
            "url": url
        })

    return articulos


def extraer_texto_jsonld(soup):
    scripts = soup.find_all("script", type="application/ld+json")

    for script in scripts:
        contenido = script.string

        if not contenido:
            continue

        try:
            data = json.loads(contenido)
        except Exception:
            continue

        candidatos = []

        if isinstance(data, dict):
            candidatos.append(data)

            if "@graph" in data and isinstance(data["@graph"], list):
                candidatos.extend(data["@graph"])

        elif isinstance(data, list):
            candidatos.extend(data)

        for item in candidatos:
            if not isinstance(item, dict):
                continue

            body = item.get("articleBody") or item.get("description")

            if body:
                body = limpiar_texto(body)

                if len(body) >= 200:
                    return body

    return ""


def extraer_parrafos_html(soup):
    selectores = [
        "article p",
        "main article p",
        "main p",
        ".entry-content p",
        ".post-content p",
        ".article-content p",
        ".article-body p",
        ".content p",
        ".texto p",
        ".noticia p",
        ".noticia-texto p",
        ".contenido p",
        "#content p",
        "#main p"
    ]

    for selector in selectores:
        encontrados = soup.select(selector)

        if len(encontrados) < 2:
            continue

        textos = []

        for p in encontrados:
            txt = limpiar_texto(p.get_text(" ", strip=True))

            if len(txt) < 35:
                continue

            if es_basura(txt):
                continue

            textos.append(txt)

        texto = " ".join(textos).strip()

        if len(texto) >= 200:
            return texto

    return ""


def extraer_todos_los_parrafos(soup):
    parrafos = []

    for p in soup.find_all("p"):
        txt = limpiar_texto(p.get_text(" ", strip=True))

        if len(txt) < 35:
            continue

        if es_basura(txt):
            continue

        parrafos.append(txt)

    return " ".join(parrafos).strip()


def extraer_articulo(url, titulo_listado=""):
    soup = pedir_sopa(url)

    h1 = soup.find("h1")
    titulo = limpiar_texto(h1.get_text(" ", strip=True)) if h1 else titulo_listado

    texto_completo = extraer_texto_jsonld(soup)

    if len(texto_completo) < 200:
        texto_completo = extraer_parrafos_html(soup)

    if len(texto_completo) < 200:
        texto_completo = extraer_todos_los_parrafos(soup)

    texto_completo = limpiar_texto(texto_completo)

    if len(texto_completo) < 200:
        return None

    if not titulo:
        titulo = "Sin título"

    return {
        "periodico": "Tercera Información",
        "titulo": titulo,
        "texto": texto_completo,
        "url": url
    }


def generar_urls_listado():
    urls = [START_URL]

    for page in range(2, MAX_PAGES + 1):
        urls.append(f"https://www.tercerainformacion.es/opinion/page/{page}/")

    return urls


def guardar_csv(datos):
    if not datos:
        return pd.DataFrame()

    df = pd.DataFrame(datos)

    if df.empty:
        return df

    df = df.drop_duplicates(subset=["url"])
    df = df[["periodico", "titulo", "texto", "url"]]

    df.to_csv(
        OUTPUT_CSV,
        index=False,
        sep="|",
        encoding="utf-8-sig"
    )

    return df


def scrape_tercerainformacion(max_items=MAX_ITEMS):
    datos = []
    articulos_vistos = set()
    paginas_sin_nuevos = 0

    urls_listado = generar_urls_listado()

    for current_url in urls_listado:
        if len(datos) >= max_items:
            break

        print(f"\nPágina listado: {current_url}")

        try:
            soup = pedir_sopa(current_url)
        except Exception as e:
            print(f"Error al descargar listado: {e}")
            paginas_sin_nuevos += 1

            if paginas_sin_nuevos >= 10:
                print("Demasiadas páginas seguidas sin resultados. Se detiene.")
                break

            continue

        articulos = extraer_urls_listado(soup, current_url)
        print("URLs encontradas:", len(articulos))

        if len(articulos) == 0:
            paginas_sin_nuevos += 1

            if paginas_sin_nuevos >= 10:
                print("Demasiadas páginas seguidas sin artículos. Se detiene.")
                break

            continue

        nuevos = 0

        for item in tqdm(articulos, desc="Entrando en artículos"):
            url = item["url"]

            if url in articulos_vistos:
                continue

            articulos_vistos.add(url)

            try:
                articulo = extraer_articulo(
                    url=url,
                    titulo_listado=item["titulo_listado"]
                )
            except Exception as e:
                print(f"Error en artículo {url}: {e}")
                continue

            if articulo is None:
                continue

            datos.append(articulo)
            nuevos += 1

            if len(datos) >= max_items:
                break

            dormir()

        print(f"Añadidos en este listado: {nuevos}")
        print(f"Total acumulado: {len(datos)}")

        if nuevos == 0:
            paginas_sin_nuevos += 1
        else:
            paginas_sin_nuevos = 0

        guardar_csv(datos)

        if paginas_sin_nuevos >= 10:
            print("Demasiadas páginas seguidas sin nuevos artículos. Se detiene.")
            break

        dormir()

    df = guardar_csv(datos)

    print(f"\nCSV guardado como: {OUTPUT_CSV}")
    print(f"Filas finales: {len(df)}")

    return df


df_tercera = scrape_tercerainformacion()
df_tercera.head()


Página listado: https://www.tercerainformacion.es/opinion/
URLs encontradas: 10


Entrando en artículos:  90%|█████████ | 9/10 [00:27<00:03,  3.05s/it]


Añadidos en este listado: 10
Total acumulado: 10

CSV guardado como: data_tercerainformacion_opinion.csv
Filas finales: 10


,periodico,titulo,texto,url
0,Tercera Información,El GUR ucraniano usa combatientes latinoameric...,David Fonseca • Opinión • 13/05/2026 Los carte...,https://www.tercerainformacion.es/opinion/13/0...
1,Tercera Información,"América Latina: No a la Guerra, No a la OTAN",Desde América del Sur observamos con mucha pre...,https://www.tercerainformacion.es/opinion/13/0...
2,Tercera Información,Abanicos de paz y esperanza,Colectivo Quejío Andalú • Opinión • 11/05/2026...,https://www.tercerainformacion.es/opinion/11/0...
3,Tercera Información,8–9 de Mayo Rusia celebra la victoria de la he...,﻿Ramón Pedregal Casanova • Opinión • 10/05/202...,https://www.tercerainformacion.es/opinion/10/0...
4,Tercera Información,A verdade sobre o modelo de MERCADONA,André Abeledo Fernández • Opinión • 10/05/2026...,https://www.tercerainformacion.es/opinion/10/0...


In [3]:
from google.colab import files

files.download("data_tercerainformacion_opinion.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>